In [0]:
%sql



In [0]:
%sql
-- CREATE SCHEMA workspace.bronzebike
-- CREATE SCHEMA workspace.silverbike
CREATE SCHEMA workspace.goldbike

In [0]:
%sql
-- CREATE VOLUME workspace.bronzebike.bronzevolume
-- CREATE VOLUME workspace.silverbike.silvervolume
CREATE VOLUME workspace.goldbike.goldvolume


In [0]:
# dbutils.fs.mkdirs("/Volumes/workspace/bronzebike/bronzevolume/bronzedata")
# dbutils.fs.mkdirs("/Volumes/workspace/silverbike/silvervolume/silverdata")
dbutils.fs.mkdirs("/Volumes/workspace/goldbike/goldvolume/golddata")



In [0]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import io

# 1. Setup Anonymous AWS Client
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

bucket_name = 'tripdata'
file_key = '202401-citibike-tripdata.zip' 
# Use the POSIX path for writing via the File API
volume_path = f"/Volumes/workspace/rawbike/rawvolume/rawdata/{file_key}"

try:
    print(f"Downloading {file_key} to memory...")
    
    # Get the object from S3
    response = s3.get_object(Bucket=bucket_name, Key=file_key)
    
    # Read the data into a memory buffer
    # Note: January 2024 is ~50-100MB, which fits easily in Community Edition RAM
    data = response['Body'].read()
    
    print(f"Streaming data to Managed Volume: {volume_path}")
    
    # Write directly to the Volume path using Python's built-in 'open'
    # Databricks allows this for Volume paths specifically
    with open(volume_path, "wb") as f:
        f.write(data)
        
    print(f"✅ Success! File saved to {volume_path}")

except Exception as e:
    print(f"❌ Error: {e}")

In [0]:
%sh
# Step 1: Copy zip FROM volume TO local driver storage (/tmp)
# This usually succeeds via FUSE even in restricted modes
cp /Volumes/workspace/rawbike/rawvolume/rawdata/202401-citibike-tripdata.zip /tmp/202401-citibike-tripdata.zip

# Check space (important on serverless — /tmp is limited)
df -h /tmp




In [0]:
%sh
# Step 2: Unzip locally
mkdir -p /tm p/extracted
unzip -o /tmp/202401-citibike-tripdata.zip -d /tmp/extracted   # -o = overwrite without asking

# Step 3: Copy extracted files BACK TO volume
# Use -r for recursive if there are subfolders; plain cp usually works here
cp -r /tmp/extracted/* /Volumes/workspace/rawbike/rawvolume/rawdata/

# Step 4: Clean up to free local space
rm -rf /tmp/myfile.zip /tmp/extracted